# 03 — PV hosting capacity

## Objectives and engineering concept

Understand that hosting capacity is a criterion-bound search, not a single universal number.

## System, assumptions, and units

The IEEE13 feeder receives a constant-power-factor PV at bus 675. The declared overvoltage limit is `1.05 pu`; the search input is kW.

## Part A — Pure OpenDSS

A short manual sweep records the maximum unregulated voltage at 0, 1000, and 2000 kW.

## Part B — Same study with CEPT

CEPT runs its typed hosting-capacity search with the same feeder and criterion.

## Part C — Compare and verify

The baseline and bracket are compared and the solver-backed result is verified. The repeated sweep is the manual pain CEPT automates; the lesson does not prove interconnection acceptance.

Hosting capacity is a search over a declared criterion. Here the criterion is overvoltage at `v_max=1.05 pu`, using a constant-power-factor PV at bus 675. We first inspect a direct OpenDSS sweep, then compare its baseline and bracket with CEPT's typed search.

## Interpret, exercise, and reproduce

Interpret the result as a criterion-specific bracket. Change the limit or candidate bus only after checking the declared units, then restart and run all cells. The final assertions and displayed package version make the run repeatable.

In [ ]:
from importlib.resources import files
from pathlib import Path
import opendssdirect as dss
from cept.public import demo_case, run_study

master = Path(str(files('cept').joinpath('testsystems', 'ieee13', 'IEEE13Nodeckt.dss')))
excluded = {'sourcebus', '650', 'rg60'}

def load_base():
    dss.Basic.ClearAll()
    dss.Basic.DataPath(str(master.parent))
    dss.Text.Command(f'Redirect "{master}"')
    dss.Text.Command('CalcVoltageBases')
    dss.Text.Command('Solve')
    assert dss.Solution.Converged()

def max_unregulated_voltage():
    names = dss.Circuit.AllNodeNames()
    values = dss.Circuit.AllBusMagPu()
    return max(value for name, value in zip(names, values) if name.split('.')[0].lower() not in excluded)

direct_sweep = {}
for kw in (0, 1000, 2000):
    load_base()
    dss.Text.Command(
        f'New PVSystem.lesson_pv phases=3 bus1=675.1.2.3 kV=4.16 '
        f'kVA={max(kw, 1)} Pmpp={kw} irradiance=1 pf=1 %cutin=0.05 %cutout=0.05'
    )
    dss.Text.Command('Solve')
    assert dss.Solution.Converged()
    direct_sweep[kw] = max_unregulated_voltage()

run = run_study(demo_case('hosting-capacity'))
hosting = run.result.hosting_capacity
assert run.verification['passed'] is True
item = next(row for row in hosting.items if row.bus.lower() == '675')
print({'direct_sweep': direct_sweep, 'cept_baseline_v_max_pu': hosting.baseline_v_max_pu, 'cept_675': item.model_dump()})
assert abs(direct_sweep[0] - hosting.baseline_v_max_pu) < 1e-3
assert direct_sweep[1000] < 1.05 < direct_sweep[2000]
assert 1000 <= item.hc_kw <= 2000


The direct sweep brackets the CEPT search result. A real interconnection decision would additionally require source-bound equipment ratings, protection settings, operating scenarios, and reviewer acceptance.